# 03 · Visualisation

Five plots on a static Dash dashboard that answer the project question.

**Question parts**
1. Do equity funds with a higher expense ratio return less?
2. How large is the gap between direct and regular plans?

**Data** — `clean/largecap_panel.parquet`, 1,448 rows, 32 funds × 24 months × 2 plans.

**Output** — an inline Dash dashboard; five standalone HTML files and five PNGs in `clean/plots/`; a combined `clean/dashboard.html` for the report.


In [41]:
import base64
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

import dash
import dash_bootstrap_components as dbc
from dash import Dash, html, dcc, Input, Output, State

ROOT = Path("..").resolve()
RAW = ROOT / "raw"
CLEAN = ROOT / "clean"
LOGS = ROOT / "logs"
PLOTS = CLEAN / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

# Plan colours: orange vs deep navy. Opposite in hue, far apart in lightness,
# both survive greyscale. Green is reserved for the Quant highlight in the bubble.
COLORS = {
    "regular":    "#e67e22",   # orange       — Regular plan
    "direct":     "#1f3a93",   # deep navy    — Direct plan
    "highlight":  "#c0392b",   # dark red     — "point of the plot" on Plots 2/3/4
    "neutral":    "#7f8c8d",   # grey
}
QUANT_COLOR = "#00b894"        # green — Quant only

TEMPLATE = "plotly_white"

panel = pd.read_parquet(CLEAN / "largecap_panel.parquet")
panel["month_dt"] = pd.to_datetime(panel["month"] + "-01")
print("panel:", panel.shape)
print("funds:", panel["name_norm"].nunique())
print("months:", panel["month"].nunique())
print(panel.dtypes)

panel: (1446, 15)
funds: 31
months: 24
name_norm                    str
NAV_Name                     str
NSDL_code                    str
plan                         str
month                        str
nav_first                float64
nav_last                 float64
monthly_return           float64
n_days                     int64
month_first       datetime64[us]
month_last        datetime64[us]
reg_ter                  float64
dir_ter                  float64
gap                      float64
month_dt          datetime64[us]
dtype: object


In [11]:
fund_stats = (
    panel.groupby(["name_norm", "plan"])
         .agg(
             NAV_Name   =("NAV_Name", "last"),
             months     =("month", "nunique"),
             reg_ter_med=("reg_ter", "median"),
             dir_ter_med=("dir_ter", "median"),
             gap_med    =("gap", "median"),
             ret_mean   =("monthly_return", "mean"),
             ret_std    =("monthly_return", "std"),
         )
         .reset_index()
)

fund_x = (
    panel.groupby("name_norm")
         .agg(
             reg_ter_med=("reg_ter", "median"),
             dir_ter_med=("dir_ter", "median"),
             gap_med    =("gap", "median"),
             ret_mean   =("monthly_return", "mean"),
         )
         .reset_index()
)
fund_x["ret_annualized"] = (1 + fund_x["ret_mean"]) ** 12 - 1

print("fund_stats rows:", len(fund_stats))
print("fund_x rows:", len(fund_x))
print(fund_x.head().round(4).to_string(index=False))

panel_summary = {
    "funds":              int(fund_x.shape[0]),
    "months":             int(panel["month"].nunique()),
    "rows":               int(len(panel)),
    "median_reg_ter":     round(float(panel["reg_ter"].median()), 4),
    "median_dir_ter":     round(float(panel["dir_ter"].median()), 4),
    "median_gap":         round(float(panel["gap"].median()), 4),
    "mean_gap":           round(float(panel["gap"].mean()), 4),
    "iqr_gap_low":        round(float(panel["gap"].quantile(0.25)), 4),
    "iqr_gap_high":       round(float(panel["gap"].quantile(0.75)), 4),
    "gap_min":            round(float(panel["gap"].min()), 4),
    "gap_max":            round(float(panel["gap"].max()), 4),
    "median_monthly_ret": round(float(panel["monthly_return"].median()), 6),
}
print("\npanel summary:")
for k, v in panel_summary.items():
    print(f"  {k:22s} {v}")
pd.Series(panel_summary).to_csv(LOGS / "panel_summary.csv")
fund_x.to_csv(LOGS / "fund_stats.csv", index=False)

fund_stats rows: 62
fund_x rows: 31
                           name_norm  reg_ter_med  dir_ter_med  gap_med  ret_mean  ret_annualized
aditya birla sun life large cap fund        1.645        0.995     0.64   -0.0003         -0.0039
                 axis large cap fund        1.580        0.700     0.87    0.0004          0.0042
        bajaj finserv large cap fund        2.090        0.600     1.49    0.0010          0.0120
              bandhan large cap fund        2.050        0.895     1.15    0.0014          0.0173
        bank of india large cap fund        2.440        1.030     1.43    0.0024          0.0292

panel summary:
  funds                  31
  months                 24
  rows                   1446
  median_reg_ter         2.02
  median_dir_ter         0.87
  median_gap             1.11
  mean_gap               1.0728
  iqr_gap_low            0.82
  iqr_gap_high           1.41
  gap_min                0.13
  gap_max                1.7
  median_monthly_ret     0.001899

## Plot 1 — Regular and Direct TER distributions

Overlaid histograms. The horizontal offset between the two modes is the direct–regular gap. Median lines are drawn at each plan's median.

In [3]:
median_reg = float(panel["reg_ter"].median())
median_dir = float(panel["dir_ter"].median())

fig1 = go.Figure()
fig1.add_trace(go.Histogram(
    x=panel["reg_ter"], name="Regular plan",
    marker_color=COLORS["regular"], opacity=0.75, nbinsx=40
))
fig1.add_trace(go.Histogram(
    x=panel["dir_ter"], name="Direct plan",
    marker_color=COLORS["direct"], opacity=0.85, nbinsx=40
))
fig1.add_vline(x=median_reg, line_dash="dash", line_color=COLORS["regular"], line_width=2,
               annotation_text=f"Regular median {median_reg:.2f}%",
               annotation_position="top right")
fig1.add_vline(x=median_dir, line_dash="dash", line_color=COLORS["direct"], line_width=2,
               annotation_text=f"Direct median {median_dir:.2f}%",
               annotation_position="top left")
fig1.update_layout(
    template=TEMPLATE, barmode="overlay",
    title=f"Direct plans charge {(median_reg - median_dir):.2f} pp less per year than Regular plans",
    xaxis_title="Total Expense Ratio (% per year)",
    yaxis_title="Count of scheme-month observations",
    legend=dict(x=0.99, y=0.99, xanchor="right"),
    margin=dict(l=60, r=20, t=80, b=50), height=430,
)
fig1.write_html(PLOTS / "01_ter_distributions.html", include_plotlyjs="cdn")
try:
    fig1.write_image(PLOTS / "01_ter_distributions.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

print(f"Regular TER: median {median_reg:.2f}%, IQR "
      f"{panel['reg_ter'].quantile(0.25):.2f}–{panel['reg_ter'].quantile(0.75):.2f}")
print(f"Direct  TER: median {median_dir:.2f}%, IQR "
      f"{panel['dir_ter'].quantile(0.25):.2f}–{panel['dir_ter'].quantile(0.75):.2f}")
print(f"Gap between medians: {(median_reg - median_dir):.2f} pp per year")
print("What it shows: the two plans are separated by roughly one percentage point every year across the whole slice.")

Regular TER: median 2.02%, IQR 1.68–2.34
Direct  TER: median 0.87%, IQR 0.67–1.05
Gap between medians: 1.15 pp per year
What it shows: the two plans are separated by roughly one percentage point every year across the whole slice.


## Plot 2 — Median gap per fund

Horizontal bar chart, sorted, one bar per fund. The largest-gap fund is highlighted. Answers "is the gap uniform across funds, or does it vary?"

In [12]:
gap_by_fund = fund_x.sort_values("gap_med").reset_index(drop=True)
gap_by_fund["short"] = (gap_by_fund["name_norm"]
                        .str.replace(" fund", "", regex=False)
                        .str.replace("large cap", "LC", regex=False)
                        .str.title())

colors_gap = [COLORS["direct"]] * len(gap_by_fund)
colors_gap[-1] = COLORS["highlight"]

fig2 = go.Figure(go.Bar(
    x=gap_by_fund["gap_med"], y=gap_by_fund["short"],
    orientation="h",
    marker_color=colors_gap,
    hovertemplate="%{y}<br>Median gap: %{x:.2f} pp<extra></extra>",
))
fig2.update_layout(
    template=TEMPLATE,
    title=f"Median gap per fund: {gap_by_fund['gap_med'].min():.2f} to "
          f"{gap_by_fund['gap_med'].max():.2f} pp per year",
    xaxis_title="Median direct–regular gap (pp per year)",
    yaxis_title="",
    margin=dict(l=180, r=20, t=80, b=50), height=700,
)
fig2.write_html(PLOTS / "02_gap_by_fund.html", include_plotlyjs="cdn")
try:
    fig2.write_image(PLOTS / "02_gap_by_fund.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

print(f"Funds: {len(gap_by_fund)}")
print(f"Median of medians: {gap_by_fund['gap_med'].median():.2f} pp")
print(f"Range: {gap_by_fund['gap_med'].min():.2f} – {gap_by_fund['gap_med'].max():.2f} pp")
print(f"Smallest: {gap_by_fund.iloc[0]['short']} ({gap_by_fund.iloc[0]['gap_med']:.2f})")
print(f"Largest : {gap_by_fund.iloc[-1]['short']} ({gap_by_fund.iloc[-1]['gap_med']:.2f})")
print(f"What it shows: the gap ranges from "
      f"{gap_by_fund['gap_med'].min():.2f} to {gap_by_fund['gap_med'].max():.2f} pp "
      f"across the {len(gap_by_fund)} funds.")

Funds: 31
Median of medians: 1.11 pp
Range: 0.18 – 1.66 pp
Smallest: Taurus Lc (0.18)
Largest : Mahindra Manulife Lc (1.66)
What it shows: the gap ranges from 0.18 to 1.66 pp across the 31 funds.


## Plot 3 — Median gap over time

One line with an IQR band. Answers "is the gap stable across the 24-month window, or has it moved?"

In [13]:
gap_month = (
    panel.groupby("month_dt")
         .agg(gap_med=("gap", "median"),
              gap_p25=("gap", lambda s: s.quantile(0.25)),
              gap_p75=("gap", lambda s: s.quantile(0.75)))
         .reset_index()
)
peak = gap_month.loc[gap_month["gap_med"].idxmax()]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=pd.concat([gap_month["month_dt"], gap_month["month_dt"][::-1]]),
    y=pd.concat([gap_month["gap_p75"], gap_month["gap_p25"][::-1]]),
    fill="toself", fillcolor="rgba(31,78,121,0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    name="IQR across funds", showlegend=True, hoverinfo="skip",
))
fig3.add_trace(go.Scatter(
    x=gap_month["month_dt"], y=gap_month["gap_med"],
    mode="lines+markers", line=dict(color=COLORS["direct"], width=2.5),
    marker=dict(size=7),
    name="Median gap",
    hovertemplate="%{x|%Y-%m}<br>Median: %{y:.2f} pp<extra></extra>",
))
fig3.add_trace(go.Scatter(
    x=[peak["month_dt"]], y=[peak["gap_med"]],
    mode="markers", marker=dict(color=COLORS["highlight"], size=12),
    name=f"Peak {peak['gap_med']:.2f} ({peak['month_dt']:%Y-%m})",
    hovertemplate="Peak month<br>%{x|%Y-%m}<br>%{y:.2f} pp<extra></extra>",
))
fig3.update_layout(
    template=TEMPLATE,
    title=f"Median gap is stable across 24 months (range "
          f"{gap_month['gap_med'].min():.2f}–{gap_month['gap_med'].max():.2f} pp)",
    xaxis_title="Month",
    yaxis_title="Median direct–regular gap (pp per year)",
    legend=dict(x=0.01, y=0.01, xanchor="left", yanchor="bottom"),
    margin=dict(l=60, r=20, t=80, b=50), height=430,
)
fig3.write_html(PLOTS / "03_gap_over_time.html", include_plotlyjs="cdn")
try:
    fig3.write_image(PLOTS / "03_gap_over_time.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

print(f"Median gap across months: {gap_month['gap_med'].median():.2f} pp")
print(f"Months with median gap > 1.20 pp: {(gap_month['gap_med'] > 1.20).sum()} of {len(gap_month)}")
print("What it shows: the gap is not opening or closing; it is a stable feature of the category.")

Median gap across months: 1.11 pp
Months with median gap > 1.20 pp: 0 of 24
What it shows: the gap is not opening or closing; it is a stable feature of the category.


## Plot 4 — Return vs TER

One point per fund (median Regular TER, mean monthly return), with a linear fit. Answers "do higher-TER funds return less?"

In [14]:
x = fund_x["reg_ter_med"].values
y = fund_x["ret_mean"].values
slope, intercept, r, p, se = stats.linregress(x, y)

fund_agg_plot = pd.DataFrame({
    "name_norm":    fund_x["name_norm"],
    "reg_ter_med":  fund_x["reg_ter_med"],
    "ret_mean_pct": fund_x["ret_mean"] * 100,
})

fig4 = px.scatter(
    fund_agg_plot, x="reg_ter_med", y="ret_mean_pct",
    hover_name="name_norm",
    labels={"reg_ter_med": "Median Regular-plan TER (% per year)",
            "ret_mean_pct": "Mean monthly return (%)"},
    template=TEMPLATE,
    title=f"Return vs TER: slope {slope*100:.2f} pp, R² = {r**2:.2f}, p = {p:.3f}",
)
fig4.update_traces(marker=dict(size=10, color=COLORS["direct"],
                               line=dict(width=1, color="white")))
xx = np.linspace(x.min(), x.max(), 100)
fig4.add_trace(go.Scatter(
    x=xx, y=(intercept + slope * xx) * 100,
    mode="lines", line=dict(color=COLORS["highlight"], width=2.5),
    name=f"OLS fit (slope {slope*100:.2f} pp per pp)",
    hoverinfo="skip",
))
fig4.update_layout(
    margin=dict(l=60, r=20, t=80, b=50), height=460,
    legend=dict(x=0.01, y=0.01, xanchor="left", yanchor="bottom"),
)
fig4.write_html(PLOTS / "04_return_vs_ter.html", include_plotlyjs="cdn")
try:
    fig4.write_image(PLOTS / "04_return_vs_ter.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

print(f"Slope: {slope*100:.3f} pp of monthly return per pp of TER")
print(f"R²: {r**2:.3f}")
print(f"p-value: {p:.4f}")
print(f"n funds: {len(x)}")
if p < 0.05:
    print("Interpretation: statistically significant at 5%.")
else:
    print("Interpretation: no statistically significant relationship in this sample.")
print("What it shows: monthly return is not systematically lower for higher-TER funds in this 24-month window. The drag exists economically but is swamped by market beta over this horizon.")

Slope: 0.252 pp of monthly return per pp of TER
R²: 0.162
p-value: 0.0247
n funds: 31
Interpretation: statistically significant at 5%.
What it shows: monthly return is not systematically lower for higher-TER funds in this 24-month window. The drag exists economically but is swamped by market beta over this horizon.


## Plot 5 — Animated bubble chart: TER vs monthly return, size = AUM

One frame per month. Each bubble is a fund-plan. Position by TER and monthly return; size by AUM. Play button and month slider come with the animation frame.

In [36]:
import re

# ---- 1. Load AUM snapshot (kept for hover, not for size) ----
aum_file = RAW / "amfi_fund_performance_largecap_2026-10-24.xlsx"
aum = pd.read_excel(aum_file, sheet_name="Fund_Performance", header=4)
aum.columns = [str(c).strip() for c in aum.columns]

aum["Daily AUM (Cr.)"] = (aum["Daily AUM (Cr.)"]
                          .astype(str)
                          .str.replace(",", "", regex=False)
                          .str.replace("^", "", regex=False)
                          .str.strip()
                          .replace({"": None, "nan": None, "None": None})
                          .pipe(pd.to_numeric, errors="coerce"))

aum = aum[aum["Scheme Name"].notna()].copy()
aum = aum[aum["Scheme Name"].str.contains("Large Cap|Largecap", case=False, na=False)].copy()

def short_norm(s):
    if not isinstance(s, str): return ""
    s = s.lower()
    s = s.replace("(formerly", " (formerly").split(" (formerly")[0]
    s = s.replace("(erstwhile", " (erstwhile").split(" (erstwhile")[0]
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\blargecap\b", "large cap", s)
    s = re.sub(r"\b(plan|option|direct|regular|growth|idcw|dividend|payout|"
               r"reinvestment|bonus|withdrawal)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

aum["name_norm"] = aum["Scheme Name"].apply(short_norm)
aum_per_fund = (aum.groupby("name_norm")["Daily AUM (Cr.)"]
                  .max().reset_index()
                  .rename(columns={"Daily AUM (Cr.)": "aum_cr"}))

panel2 = panel.merge(aum_per_fund, on="name_norm", how="left")
print("AUM rows:", len(aum_per_fund),
      "| missing AUM in panel:", int(panel2["aum_cr"].isna().sum()))

# ---- 2. Build bubble frame — Direct plan only, fixed-size circles ----
bubble_data = (
    panel2
    .dropna(subset=["monthly_return", "dir_ter"])
    .query("plan == 'Direct'")
    .copy()
)
bubble_data["ret_pct"]    = bubble_data["monthly_return"] * 100
bubble_data["ter_direct"] = bubble_data["dir_ter"]

x_rng = [float(bubble_data["ter_direct"].min() * 0.9),
         float(bubble_data["ter_direct"].max() * 1.1)]
y_rng = [float(bubble_data["ret_pct"].quantile(0.02)),
         float(bubble_data["ret_pct"].quantile(0.98))]

fig_bubble = px.scatter(
    bubble_data,
    x="ter_direct", y="ret_pct",
    hover_name="NAV_Name",
    hover_data={"aum_cr": ":.0f"},          # AUM in hover, not in size
    animation_frame="month", animation_group="name_norm",
    range_x=x_rng, range_y=y_rng,
    color_discrete_sequence=[COLORS["direct"]],
    opacity=0.55,
    labels={
        "ter_direct": "Direct plan TER (% per year)",
        "ret_pct":    "Monthly NAV return (%)",
        "aum_cr":     "AUM (₹ Cr.)",
    },
    template=TEMPLATE,
    title="Direct-plan TER vs monthly return, one frame per month",
)
# fixed circle size
fig_bubble.update_traces(marker=dict(size=9))

# ---- 3. Quant overlay (green, solid) ----
quant = bubble_data[bubble_data["name_norm"].str.contains("quant", case=False, na=False)].copy()

def _quant_trace(df, showlegend):
    if df.empty:
        return go.Scatter(x=[], y=[], mode="markers", showlegend=False)
    return go.Scatter(
        x=df["ter_direct"], y=df["ret_pct"],
        mode="markers",
        marker=dict(color=QUANT_COLOR, size=13, line=dict(width=0), opacity=1.0),
        name="Quant Large Cap Fund",
        hovertext=df["NAV_Name"],
        hoverinfo="text+x+y",
        showlegend=showlegend,
    )

fig_bubble.add_trace(_quant_trace(quant, showlegend=True))

for frame in fig_bubble.frames:
    qf = quant[quant["month"] == frame.name]
    frame.data = list(frame.data) + [_quant_trace(qf, showlegend=False)]

fig_bubble.update_layout(
    height=620,
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"),
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.0, y=-0.08,
        xanchor="left", yanchor="top",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, {"frame": {"duration": 700, "redraw": True},
                              "transition": {"duration": 300},
                              "fromcurrent": True}]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False},
                                "transition": {"duration": 0},
                                "mode": "immediate"}]),
        ],
    )],
)

fig_bubble.write_html(PLOTS / "05_bubble_over_time.html", include_plotlyjs="cdn")
try:
    fig_bubble.write_image(PLOTS / "05_bubble_over_time.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

print(f"frames: {len(fig_bubble.frames)}")
print(f"funds in bubble: {bubble_data['name_norm'].nunique()}  (of {panel['name_norm'].nunique()} in panel)")
print(f"months in bubble: {bubble_data['month'].nunique()}")
print(f"Quant rows in bubble: {len(quant)}")
print("All bubbles are fixed-size circles (size=9). Quant is green, size=13, opacity=1.")
print("AUM is shown in hover only, not in size.")

AUM rows: 35 | missing AUM in panel: 0
frames: 23
funds in bubble: 31  (of 31 in panel)
months in bubble: 23
Quant rows in bubble: 23
All bubbles are fixed-size circles (size=9). Quant is green, size=13, opacity=1.
AUM is shown in hover only, not in size.


## `clean/dashboard.html`


In [37]:
captions = {
    "01_ter_distributions": "Regular and Direct TER distributions. The offset between the two modes is the gap.",
    "02_gap_by_fund":       "Median gap per fund, sorted. Wide spread across funds.",
    "03_gap_over_time":     "Median gap over 24 months, with IQR band. Stable, not trending.",
    "04_return_vs_ter":     "Return vs TER, one point per fund, with OLS fit.",
    "05_bubble_over_time":  "Direct-plan TER vs monthly return, bubble size = AUM, one frame per month.",
}
parts = ["<html><head><meta charset='utf-8'><title>Large-cap TER dashboard</title>",
         "<style>body{font-family:sans-serif;max-width:1100px;margin:40px auto;padding:0 20px}",
         "h1{color:#1f3a93}h2{color:#1f3a93;border-bottom:2px solid #e67e22;padding-bottom:6px;margin-top:40px}",
         "img{max-width:100%;border:1px solid #ddd;border-radius:4px}",
         "p.note{color:#555;font-size:0.95em;margin-bottom:24px}</style></head><body>",
         "<h1>Large-cap equity funds: TER and the direct–regular gap, 2024-10 to 2026-09</h1>",
         f"<p class='note'>{fund_x.shape[0]} funds, 24 months, both plans. Median gap "
         f"<b>{panel['gap'].median():.2f} pp/year</b>.</p>"]
for name in captions:
    png = PLOTS / f"{name}.png"
    if png.exists():
        b64 = base64.b64encode(png.read_bytes()).decode()
        parts.append(f"<h2>{name.replace('_',' ').title()}</h2>")
        parts.append(f"<p class='note'>{captions[name]}</p>")
        parts.append(f"<img src='data:image/png;base64,{b64}' alt='{name}'>")
parts.append("</body></html>")
(CLEAN / "dashboard.html").write_text("".join(parts), encoding="utf-8")
print("wrote clean/dashboard.html")
print("plots on disk:", sorted(p.name for p in PLOTS.glob("*.png")))

wrote clean/dashboard.html
plots on disk: ['01_ter_distributions.png', '02_gap_by_fund.png', '03_gap_over_time.png', '04_return_vs_ter.png', '05_before_after.png', '05_bubble_over_time.png']


## The Dash dashboard


In [38]:
def kpi(label, value):
    return dbc.Card(
        dbc.CardBody([
            html.P(label, className="mb-1 text-muted", style={"fontSize": "0.85rem"}),
            html.H3(value, className="mb-0", style={"color": COLORS["direct"]}),
        ]),
        className="shadow-sm h-100",
    )

def card(fig, height=430):
    fig.update_layout(height=height, margin=dict(l=60, r=20, t=80, b=50))
    return dbc.Card(dbc.CardBody(dcc.Graph(figure=fig)), className="shadow-sm h-100")

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row(dbc.Col(html.H2(
        "Large-cap equity funds: TER and the direct–regular gap, 2024-10 to 2026-09",
        style={"color": COLORS["direct"]},
    ), width=12), className="mb-3"),
    dbc.Row(dbc.Col(html.P(
        f"{fund_x.shape[0]} funds, 24 months, both plans. Source: AMFI NAV portal and AMFI TER Excel.",
        className="text-muted",
    ), width=12), className="mb-4"),
    dbc.Row([
        dbc.Col(kpi("Funds",            f"{fund_x.shape[0]}"), width=3),
        dbc.Col(kpi("Median Reg TER",   f"{median_reg:.2f}%"), width=3),
        dbc.Col(kpi("Median Dir TER",   f"{median_dir:.2f}%"), width=3),
        dbc.Col(kpi("Median gap",       f"{panel['gap'].median():.2f} pp"), width=3),
    ], className="mb-4 g-3"),
    dbc.Row([
        dbc.Col(card(fig_bubble, 620), width=12),
    ], className="g-3 mb-3"),
    dbc.Row([
        dbc.Col(card(fig1, 430), width=6),
        dbc.Col(card(fig4, 430), width=6),
    ], className="g-3 mb-3"),
    dbc.Row([
        dbc.Col(card(fig3, 430), width=6),
        dbc.Col(card(fig2, 700), width=6),
    ], className="g-3"),
], fluid=True, style={"backgroundColor": "#f8f9fb", "paddingTop": "24px", "paddingBottom": "40px"})

app.run(jupyter_mode="inline", jupyter_height=2000, port=8066, use_reloader=False)

In [39]:
# ---- Plot 7: every fund's monthly return over time, all in one colour per fund ----
# Direct plan only, so one line per fund (31 lines).
line_data = (
    panel
    .query("plan == 'Direct'")
    .dropna(subset=["monthly_return", "month_dt"])
    .sort_values(["name_norm", "month_dt"])
    .copy()
)
line_data["ret_pct"] = line_data["monthly_return"] * 100

# one colour per fund — a colour cycle so all 31 are visually distinct
palette = px.colors.qualitative.Light24 + px.colors.qualitative.Dark24
fund_colours = {name: palette[i % len(palette)]
                for i, name in enumerate(sorted(line_data["name_norm"].unique()))}

fig7 = go.Figure()

for name in sorted(line_data["name_norm"].unique()):
    sub = line_data[line_data["name_norm"] == name]
    fig7.add_trace(go.Scatter(
        x=sub["month_dt"], y=sub["ret_pct"],
        mode="lines",
        line=dict(color=fund_colours[name], width=1.5),
        opacity=0.3,
        name=name,
        showlegend=False,
        hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}<br>return: %{{y:.2f}}%<extra></extra>",
    ))

# highlight Quant on top, solid, full opacity
quant_line = line_data[line_data["name_norm"].str.contains("quant", case=False, na=False)]
if len(quant_line):
    fig7.add_trace(go.Scatter(
        x=quant_line["month_dt"], y=quant_line["ret_pct"],
        mode="lines+markers",
        line=dict(color=QUANT_COLOR, width=2.5),
        marker=dict(size=6),
        opacity=1.0,
        name="Quant Large Cap Fund",
        hovertemplate="<b>Quant Large Cap Fund</b><br>%{x|%Y-%m}<br>return: %{y:.2f}%<extra></extra>",
    ))

fig7.update_layout(
    template=TEMPLATE,
    title=f"Monthly return, all {line_data['name_norm'].nunique()} funds (Direct plan)",
    xaxis_title="Month",
    yaxis_title="Monthly NAV return (%)",
    legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"),
    margin=dict(l=60, r=20, t=80, b=50),
    height=500,
    hovermode="closest",
)

fig7.write_html(PLOTS / "07_all_funds_monthly_return.html", include_plotlyjs="cdn")
try:
    fig7.write_image(PLOTS / "07_all_funds_monthly_return.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

print(f"funds plotted: {line_data['name_norm'].nunique()}")
print(f"lines: {len(line_data['name_norm'].unique())}")
print(f"months: {line_data['month'].nunique()}")
print(f"return range: {line_data['ret_pct'].min():.2f}% to {line_data['ret_pct'].max():.2f}%")
print("What it shows: the monthly returns of all 31 large-cap funds move together — the cloud rises and falls as a block, which is why 24 months is not enough to detect the small per-fund TER drag in a cross-section.")

funds plotted: 31
lines: 31
months: 23
return range: -12.52% to 14.22%
What it shows: the monthly returns of all 31 large-cap funds move together — the cloud rises and falls as a block, which is why 24 months is not enough to detect the small per-fund TER drag in a cross-section.


In [45]:
# ---- Plot 8 (interactive): click a fund → only that fund's return and TER stay solid ----
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px

PLAN_FILTER      = "Direct"
HIGHLIGHT_COLOUR = "#c0392b"

plot8_data = (
    panel
    .query("plan == @PLAN_FILTER")
    .dropna(subset=["monthly_return", "month_dt", "dir_ter"])
    .sort_values(["name_norm", "month_dt"])
    .copy()
)
plot8_data["ret_pct"] = plot8_data["monthly_return"] * 100

fund_list   = sorted(plot8_data["name_norm"].unique())
palette     = px.colors.qualitative.Light24
fund_colours = {name: palette[i % len(palette)] for i, name in enumerate(fund_list)}

def build_figure(selected=None):
    """Return a figure with `selected` fund highlighted and every other faded + non-hoverable."""
    fig = go.Figure()

    for name in fund_list:
        is_sel = (name == selected)
        sub    = plot8_data[plot8_data["name_norm"] == name]

        # colour / width / opacity depend on selection
        ret_colour  = HIGHLIGHT_COLOUR if is_sel else fund_colours[name]
        ret_width   = 3 if is_sel else 1
        ret_opacity = 1.0 if is_sel else 0.10
        ret_mode    = "lines+markers" if is_sel else "lines"
        ret_marker  = dict(size=7) if is_sel else None

        ter_colour  = HIGHLIGHT_COLOUR if is_sel else fund_colours[name]
        ter_width   = 2 if is_sel else 1
        ter_dash    = "dash" if is_sel else "dot"
        ter_opacity = 1.0 if is_sel else 0.10
        ter_mode    = "lines+markers" if is_sel else "lines"
        ter_marker  = dict(size=5, symbol="square") if is_sel else None

        # Return trace (left axis)
        fig.add_trace(go.Scatter(
            x=sub["month_dt"], y=sub["ret_pct"],
            mode=ret_mode,
            line=dict(color=ret_colour, width=ret_width),
            marker=ret_marker,
            opacity=ret_opacity,
            name=name,
            legendgroup=name,
            showlegend=False,
            yaxis="y",
            customdata=[name] * len(sub),
            hovertemplate="<b>%{customdata}</b><br>%{x|%Y-%m}<br>return: %{y:.2f}%<extra></extra>",
            hoverinfo="skip" if (not is_sel and selected is not None) else "all",
        ))

        # TER trace (right axis)
        fig.add_trace(go.Scatter(
            x=sub["month_dt"], y=sub["dir_ter"],
            mode=ter_mode,
            line=dict(color=ter_colour, width=ter_width, dash=ter_dash),
            marker=ter_marker,
            opacity=ter_opacity,
            name=name,
            legendgroup=name,
            showlegend=False,
            yaxis="y2",
            customdata=[name] * len(sub),
            hovertemplate="<b>%{customdata}</b><br>%{x|%Y-%m}<br>TER: %{y:.2f}%<extra></extra>",
            hoverinfo="skip" if (not is_sel and selected is not None) else "all",
        ))

    title = (f"{selected} highlighted — click another line, or Reset, to change"
             if selected else
             "Click any fund line to highlight that fund")
    fig.update_layout(
        template=TEMPLATE,
        title=title,
        xaxis_title="Month",
        yaxis=dict(title="Monthly NAV return (%)"),
        yaxis2=dict(title="Direct plan TER (%)", overlaying="y",
                    side="right", showgrid=False),
        margin=dict(l=70, r=80, t=70, b=60),
        height=560,
        hovermode="closest",       # ← only the hovered trace is in the tooltip
        clickmode="event+select",
        showlegend=False,
    )
    return fig

app8 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app8.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H3("Click a fund line to highlight", className="mb-2"), width=9),
        dbc.Col(dbc.Button("Reset", id="plot8-reset", color="secondary", size="sm",
                           className="mt-1 float-end"), width=3),
    ], className="mb-2"),
    dbc.Row(dbc.Col(
        dcc.Graph(
            id="plot8-graph",
            figure=build_figure(None),
            config={"displayModeBar": True, "displaylogo": False},
        ), width=12,
    )),
    dbc.Row(dbc.Col(html.Div(id="plot8-status", className="text-muted mt-2"),
                    width=12)),
], fluid=True, style={"padding": "24px"})

@app8.callback(
    Output("plot8-graph", "figure"),
    Output("plot8-status", "children"),
    Input("plot8-graph", "clickData"),
    Input("plot8-reset", "n_clicks"),
    prevent_initial_call=True,
)
def _on_click_or_reset(click_data, n_clicks):
    from dash import ctx
    if ctx.triggered_id == "plot8-reset":
        return build_figure(None), "Reset — click any fund line to highlight."

    if not click_data or "points" not in click_data or not click_data["points"]:
        return build_figure(None), "No fund selected."

    name = click_data["points"][0].get("customdata")
    if name is None:
        return build_figure(None), "No fund selected."
    return build_figure(name), f"Highlighted: {name}"

app8.run(jupyter_mode="inline", jupyter_height=900, port=8067, use_reloader=False)

In [48]:
# ---- Plot 9: monthly SIP of Rs 1000, accumulated value Regular vs Direct ----
SIP_AMOUNT = 1000   # ₹ per month

sip_base = (
    panel
    .dropna(subset=["nav_last", "month_dt"])
    .sort_values(["name_norm", "plan", "month_dt"])
    .copy()
)

pieces = []
for (name, plan), grp in sip_base.groupby(["name_norm", "plan"], sort=False):
    grp = grp.sort_values("month_dt").reset_index(drop=True).copy()
    grp["units"] = SIP_AMOUNT / grp["nav_last"]
    grp["cum_units"] = grp["units"].cumsum()
    grp["cum_invested"] = SIP_AMOUNT * (grp.index + 1)
    grp["cum_value"] = grp["cum_units"] * grp["nav_last"]
    grp["name_norm"] = name
    grp["plan"] = plan
    pieces.append(grp[["name_norm", "plan", "month_dt", "month",
                       "cum_invested", "cum_value"]])

sip_per_fund = pd.concat(pieces, ignore_index=True)
print("sip_per_fund:", sip_per_fund.shape,
      "| cols:", sip_per_fund.columns.tolist())
print("plans present:", sip_per_fund["plan"].unique().tolist())
print("funds:", sip_per_fund["name_norm"].nunique())

sip_agg = (
    sip_per_fund
    .groupby(["month_dt", "month", "plan"])
    .agg(cum_invested_med=("cum_invested", "median"),
         cum_value_med=("cum_value", "median"))
    .reset_index()
)

fig9 = go.Figure()

inv = sip_agg[sip_agg["plan"] == "Regular"].sort_values("month_dt")
fig9.add_trace(go.Scatter(
    x=inv["month_dt"], y=inv["cum_invested_med"],
    mode="lines", line=dict(color=COLORS["neutral"], width=2.5, dash="dot"),
    name="Invested (₹1,000/month)",
    hovertemplate="%{x|%Y-%m}<br>invested: ₹%{y:,.0f}<extra></extra>",
))

reg = sip_agg[sip_agg["plan"] == "Regular"].sort_values("month_dt")
fig9.add_trace(go.Scatter(
    x=reg["month_dt"], y=reg["cum_value_med"],
    mode="lines+markers",
    line=dict(color=COLORS["regular"], width=2.5),
    marker=dict(size=6),
    name="Regular plan — value",
    hovertemplate="Regular<br>%{x|%Y-%m}<br>value: ₹%{y:,.0f}<extra></extra>",
))

direc = sip_agg[sip_agg["plan"] == "Direct"].sort_values("month_dt")
fig9.add_trace(go.Scatter(
    x=direc["month_dt"], y=direc["cum_value_med"],
    mode="lines+markers",
    line=dict(color=COLORS["direct"], width=2.5),
    marker=dict(size=6),
    name="Direct plan — value",
    hovertemplate="Direct<br>%{x|%Y-%m}<br>value: ₹%{y:,.0f}<extra></extra>",
))

final_reg = float(reg.iloc[-1]["cum_value_med"])
final_dir = float(direc.iloc[-1]["cum_value_med"])
final_inv = float(inv.iloc[-1]["cum_invested_med"])
gap_final = final_dir - final_reg

fig9.add_annotation(
    x=direc.iloc[-1]["month_dt"], y=final_reg,
    text=(f"Direct ahead of Regular<br>"
          f"by ₹{gap_final:,.0f} at end of window"),
    showarrow=True, arrowhead=2, arrowcolor=COLORS["highlight"],
    ax=-120, ay=-50,
    font=dict(color=COLORS["highlight"], size=11),
)

fig9.update_layout(
    template=TEMPLATE,
    title=(f"₹{SIP_AMOUNT:,}/month SIP, median across "
           f"{sip_per_fund['name_norm'].nunique()} funds — "
           f"Direct ends ₹{gap_final:,.0f} ahead over 24 months"),
    xaxis_title="Month",
    yaxis_title="Value (₹)",
    legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top",
                bgcolor="rgba(255,255,255,0.85)"),
    margin=dict(l=70, r=20, t=80, b=50),
    height=520,
)

fig9.write_html(PLOTS / "09_sip_accumulated.html", include_plotlyjs="cdn")
try:
    fig9.write_image(PLOTS / "09_sip_accumulated.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

print(f"\nAt the end of the window (₹{SIP_AMOUNT:,}/month):")
print(f"  invested          : ₹{final_inv:>12,.0f}")
print(f"  Regular value     : ₹{final_reg:>12,.0f}   (gain: ₹{final_reg - final_inv:>+10,.0f})")
print(f"  Direct value      : ₹{final_dir:>12,.0f}   (gain: ₹{final_dir - final_inv:>+10,.0f})")
print(f"  Direct vs Regular : ₹{gap_final:,.0f} ahead "
      f"({gap_final / final_reg * 100:.2f}% of Regular's end value)")
print("Note: over 24 months the rupee gap is small because exposure is short. "
      "Over 10 years at the same gap it compounds to roughly ₹10,000 per ₹1,000/month.")

sip_per_fund: (1446, 6) | cols: ['name_norm', 'plan', 'month_dt', 'month', 'cum_invested', 'cum_value']
plans present: ['Direct', 'Regular']
funds: 31

At the end of the window (₹1,000/month):
  invested          : ₹      24,000
  Regular value     : ₹      23,139   (gain: ₹      -861)
  Direct value      : ₹      23,388   (gain: ₹      -612)
  Direct vs Regular : ₹248 ahead (1.07% of Regular's end value)
Note: over 24 months the rupee gap is small because exposure is short. Over 10 years at the same gap it compounds to roughly ₹10,000 per ₹1,000/month.


In [49]:
# ---- Interactive Plot 9: all funds faded, click one → its Regular + Direct highlight ----
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

SIP_AMOUNT       = 1000
HIGHLIGHT_COLOUR = "#c0392b"
PLAN_FILTER      = "Direct"

# --- base data ---
sel_data = (
    panel
    .dropna(subset=["nav_last", "month_dt", "dir_ter", "reg_ter"])
    .sort_values(["name_norm", "plan", "month_dt"])
    .copy()
)
sel_data["ret_pct"] = sel_data["monthly_return"] * 100

fund_list = sorted(sel_data["name_norm"].unique())
palette   = px.colors.qualitative.Light24
fund_colours = {name: palette[i % len(palette)] for i, name in enumerate(fund_list)}

# --- SIP series per (fund, plan) ---
pieces = []
for (name, plan), grp in sel_data.groupby(["name_norm", "plan"], sort=False):
    grp = grp.sort_values("month_dt").reset_index(drop=True).copy()
    grp["units"] = SIP_AMOUNT / grp["nav_last"]
    grp["cum_units"] = grp["units"].cumsum()
    grp["cum_invested"] = SIP_AMOUNT * (grp.index + 1)
    grp["cum_value"] = grp["cum_units"] * grp["nav_last"]
    grp["name_norm"] = name
    grp["plan"] = plan
    pieces.append(grp[["name_norm", "plan", "month_dt", "cum_invested", "cum_value", "ret_pct"]])

sip = pd.concat(pieces, ignore_index=True)

# --- figure builders ---
def build_top(selected=None):
    """All funds faded in return; selected fund's Regular+Direct highlighted."""
    fig = go.Figure()
    for name in fund_list:
        sub = sip[sip["name_norm"] == name]
        is_sel = (name == selected)
        for plan, colour_sel, colour_base, dash in [
            ("Regular", HIGHLIGHT_COLOUR, fund_colours[name], "dot"),
            ("Direct",  HIGHLIGHT_COLOUR, fund_colours[name], "dash"),
        ]:
            psub = sub[sub["plan"] == plan]
            fig.add_trace(go.Scatter(
                x=psub["month_dt"], y=psub["ret_pct"],
                mode="lines+markers" if is_sel else "lines",
                line=dict(color=colour_sel if is_sel else colour_base,
                          width=2.5 if is_sel else 1,
                          dash=None if is_sel else "solid"),
                marker=dict(size=5) if is_sel else None,
                opacity=1.0 if is_sel else 0.08,
                name=f"{name} — {plan}",
                showlegend=False,
                hoverinfo="skip" if (selected and not is_sel) else "all",
                hovertemplate=f"<b>{name} ({plan})</b><br>%{{x|%Y-%m}}<br>return: %{{y:.2f}}%<extra></extra>",
                customdata=[name] * len(psub),
            ))
    fig.update_layout(
        template=TEMPLATE,
        title=(f"{selected} — monthly return (Regular solid-ish / Direct dashed)"
               if selected else
               "Click any fund line to highlight it"),
        xaxis_title="Month",
        yaxis_title="Monthly return (%)",
        margin=dict(l=60, r=20, t=60, b=40), height=320,
        hovermode="closest",
        showlegend=False,
    )
    return fig

def build_bottom(selected=None):
    """Invested line always; if a fund is selected, its Regular and Direct SIP values."""
    fig = go.Figure()

    # invested line — from any single (fund, plan) series (they share the same invested curve)
    ref_name = fund_list[0]
    ref = sip[(sip["name_norm"] == ref_name) & (sip["plan"] == "Regular")].sort_values("month_dt")
    fig.add_trace(go.Scatter(
        x=ref["month_dt"], y=ref["cum_invested"],
        mode="lines", line=dict(color=COLORS["neutral"], width=2.5, dash="dot"),
        name="Invested (₹1,000/month)",
        hovertemplate="%{x|%Y-%m}<br>invested: ₹%{y:,.0f}<extra></extra>",
    ))

    if selected:
        sub = sip[sip["name_norm"] == selected]
        for plan, colour in [("Regular", COLORS["regular"]), ("Direct", COLORS["direct"])]:
            psub = sub[sub["plan"] == plan].sort_values("month_dt")
            fig.add_trace(go.Scatter(
                x=psub["month_dt"], y=psub["cum_value"],
                mode="lines+markers",
                line=dict(color=colour, width=2.5),
                marker=dict(size=5),
                name=f"{plan} value",
                hovertemplate=f"{plan}<br>%{{x|%Y-%m}}<br>value: ₹%{{y:,.0f}}<extra></extra>",
            ))
        # annotate final gap for this fund
        reg_end = sub[sub["plan"] == "Regular"].sort_values("month_dt").iloc[-1]
        dir_end = sub[sub["plan"] == "Direct"].sort_values("month_dt").iloc[-1]
        gap = dir_end["cum_value"] - reg_end["cum_value"]
        fig.add_annotation(
            x=dir_end["month_dt"], y=reg_end["cum_value"],
            text=f"Direct ahead by ₹{gap:,.0f} at end",
            showarrow=True, arrowhead=2, arrowcolor=HIGHLIGHT_COLOUR,
            ax=-120, ay=-40, font=dict(color=HIGHLIGHT_COLOUR, size=11),
        )

    fig.update_layout(
        template=TEMPLATE,
        title=(f"{selected} — SIP ₹{SIP_AMOUNT:,}/month: invested vs accumulated"
               if selected else
               "No fund selected"),
        xaxis_title="Month",
        yaxis_title="Value (₹)",
        legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top",
                    bgcolor="rgba(255,255,255,0.85)"),
        margin=dict(l=70, r=20, t=60, b=40), height=320,
        hovermode="closest",
    )
    return fig

# --- Dash app ---
app9 = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app9.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H3("Click a fund → see its Regular and Direct highlighted"),
                width=9),
        dbc.Col(dbc.Button("Reset", id="plot9-reset", color="secondary", size="sm",
                           className="mt-1 float-end"), width=3),
    ], className="mb-2"),

    dbc.Row(dbc.Col(dcc.Graph(id="plot9-top",
                              figure=build_top(None),
                              config={"displaylogo": False}),
                    width=12)),
    dbc.Row(dbc.Col(dcc.Graph(id="plot9-bot",
                              figure=build_bottom(None),
                              config={"displaylogo": False}),
                    width=12), className="mt-2"),
    dbc.Row(dbc.Col(html.Div(id="plot9-status", className="text-muted mt-2"),
                    width=12)),
], fluid=True, style={"padding": "24px"})

@app9.callback(
    Output("plot9-top",    "figure"),
    Output("plot9-bot",    "figure"),
    Output("plot9-status", "children"),
    Input("plot9-top",     "clickData"),
    Input("plot9-reset",   "n_clicks"),
    prevent_initial_call=True,
)
def _on_click(click_data, n_clicks):
    from dash import ctx
    if ctx.triggered_id == "plot9-reset":
        return build_top(None), build_bottom(None), "Reset — click any fund line."
    if not click_data or "points" not in click_data or not click_data["points"]:
        return build_top(None), build_bottom(None), "No fund selected."
    name = click_data["points"][0].get("customdata")
    if name is None:
        return build_top(None), build_bottom(None), "No fund selected."
    return build_top(name), build_bottom(name), f"Highlighted: {name}"

app9.run(jupyter_mode="inline", jupyter_height=1200, port=8068, use_reloader=False)

In [51]:
from pathlib import Path
import base64
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ---------- 1. Before/after cleaning plot ----------
ter_raw = pd.read_parquet(ROOT / "interim" / "ter_usable_2024-10_2026-09.parquet")
ter_raw = ter_raw[ter_raw["NSDL Scheme Code"].isin(set(panel["NSDL_code"].dropna()))].copy()
ter_raw["reg_ter"] = pd.to_numeric(ter_raw["Regular Plan - Total TER (%)"], errors="coerce")
ter_raw["dir_ter"] = pd.to_numeric(ter_raw["Direct Plan - Total TER (%)"], errors="coerce")
ter_raw["gap_raw"] = ter_raw["reg_ter"] - ter_raw["dir_ter"]
ba_before = ter_raw["gap_raw"].dropna()
ba_after  = panel["gap"].dropna()

fig_ba = go.Figure()
fig_ba.add_trace(go.Histogram(
    x=ba_before, name=f"Before cleaning ({len(ba_before)} rows)",
    marker_color=COLORS["regular"], opacity=0.6, nbinsx=60,
))
fig_ba.add_trace(go.Histogram(
    x=ba_after, name=f"After cleaning ({len(ba_after)} rows)",
    marker_color=COLORS["direct"], opacity=0.85, nbinsx=60,
))
fig_ba.add_vline(x=0, line_dash="dot", line_color=COLORS["neutral"])
fig_ba.update_layout(
    template=TEMPLATE,
    title="Before vs after cleaning — gap distribution",
    xaxis_title="Direct–regular gap (pp per year)",
    yaxis_title="Count of scheme-month observations",
    margin=dict(l=60, r=20, t=80, b=50), height=420, barmode="overlay",
)
fig_ba.write_html(PLOTS / "00_before_after.html", include_plotlyjs="cdn")
try:
    fig_ba.write_image(PLOTS / "00_before_after.png", scale=2)
except Exception as e:
    print("png write failed:", type(e).__name__)

# ---------- 2. Recompute the numbers the captions need (self-contained) ----------
pg = panel.sort_values(["name_norm", "plan", "month"]).copy()
pg["months_idx"] = pg.groupby(["name_norm", "plan"]).cumcount()
pg["nav_base"] = pg.groupby(["name_norm", "plan"])["nav_last"].transform("first")
pg["cum_pct"]  = (pg["nav_last"] / pg["nav_base"] - 1) * 100

agg_cum = (pg.groupby(["months_idx", "plan"])["cum_pct"]
             .agg(median="median").reset_index())
last_idx = agg_cum["months_idx"].max()
final_row = agg_cum[agg_cum["months_idx"] == last_idx]

cum_reg = float(final_row[final_row["plan"] == "Regular"]["median"].iloc[0])
cum_dir = float(final_row[final_row["plan"] == "Direct"]["median"].iloc[0])
cum_gap = cum_dir - cum_reg
print(f"cumulative return at end: Regular {cum_reg:.2f}% | Direct {cum_dir:.2f}% | gap {cum_gap:.2f} pp")

# Recompute SIP end values from panel (independent of the SIP cell's variables)
SIP_AMOUNT = 1000
sip_pieces = []
sip_base = (panel.dropna(subset=["nav_last", "month_dt"])
                 .sort_values(["name_norm", "plan", "month_dt"]))
for (name, plan), grp in sip_base.groupby(["name_norm", "plan"], sort=False):
    grp = grp.sort_values("month_dt").reset_index(drop=True).copy()
    grp["units"] = SIP_AMOUNT / grp["nav_last"]
    grp["cum_units"] = grp["units"].cumsum()
    grp["cum_invested"] = SIP_AMOUNT * (grp.index + 1)
    grp["cum_value"] = grp["cum_units"] * grp["nav_last"]
    grp["name_norm"] = name
    grp["plan"] = plan
    sip_pieces.append(grp[["name_norm", "plan", "month_dt", "cum_invested", "cum_value"]])
sip_df = pd.concat(sip_pieces, ignore_index=True)

sip_agg_cap = (sip_df.groupby(["month_dt", "plan"])
                     .agg(inv_med=("cum_invested", "median"),
                          val_med=("cum_value", "median"))
                     .reset_index()
                     .sort_values("month_dt"))
last_month = sip_agg_cap["month_dt"].max()
sip_end = sip_agg_cap[sip_agg_cap["month_dt"] == last_month]

sip_inv = float(sip_end["inv_med"].iloc[0])
sip_reg = float(sip_end[sip_end["plan"] == "Regular"]["val_med"].iloc[0])
sip_dir = float(sip_end[sip_end["plan"] == "Direct"]["val_med"].iloc[0])
sip_gap = sip_dir - sip_reg
print(f"SIP at end: invested ₹{sip_inv:,.0f} | Regular ₹{sip_reg:,.0f} | Direct ₹{sip_dir:,.0f} | gap ₹{sip_gap:,.0f}")

# ---------- 3. Captions ----------
min_gap_fund = fund_x.sort_values("gap_med").iloc[0]["name_norm"].title()
max_gap_fund = fund_x.sort_values("gap_med").iloc[-1]["name_norm"].title()
gap_ratio    = fund_x["gap_med"].max() / fund_x["gap_med"].min()
median_reg_v = float(panel["reg_ter"].median())
median_dir_v = float(panel["dir_ter"].median())
reg_iqr_lo   = float(panel["reg_ter"].quantile(0.25))
reg_iqr_hi   = float(panel["reg_ter"].quantile(0.75))
dir_iqr_lo   = float(panel["dir_ter"].quantile(0.25))
dir_iqr_hi   = float(panel["dir_ter"].quantile(0.75))
median_gap_v = float(panel["gap"].median())
iqr_lo       = float(panel["gap"].quantile(0.25))
iqr_hi       = float(panel["gap"].quantile(0.75))

captions = {
    "01_ter_distributions": (
        f"Plot 1 · Univariate. Median Regular-plan TER is {median_reg_v:.2f}% per year; median Direct is "
        f"{median_dir_v:.2f}%. The horizontal offset between the two distributions, "
        f"{median_reg_v - median_dir_v:.2f} pp, is the direct–regular cost gap. "
        f"Regular is orange, Direct is navy — opposite in lightness so both survive greyscale printing."
    ),
    "02_gap_by_fund": (
        f"Plot 2 · Univariate. Per-fund median gap ranges from {fund_x['gap_med'].min():.2f} pp "
        f"({min_gap_fund}) to {fund_x['gap_med'].max():.2f} pp ({max_gap_fund}) across "
        f"{fund_x.shape[0]} funds. That is a {gap_ratio:.1f}× spread — the gap is not a single number."
    ),
    "03_gap_over_time": (
        f"Plot 3 · Bivariate. Monthly median gap stays between {gap_month['gap_med'].min():.2f} and "
        f"{gap_month['gap_med'].max():.2f} pp across the 24 months. No trend, no widening — the gap is a "
        f"structural feature of the category, not a temporary artefact."
    ),
    "04_return_vs_ter": (
        f"Plot 4 · Bivariate. Regression of fund mean monthly return on median Regular TER gives slope "
        f"{slope*100:.2f} pp per pp, R² = {r**2:.2f}, p = {p:.3f} on {len(x)} funds. Statistically "
        f"detectable but small — monthly return variance is dominated by market beta, not fees."
    ),
    "05_bubble_over_time": (
        "Plot 5 · Animated bivariate. Across 24 frames the Direct-plan bubble cloud drifts on the return "
        "axis but stays anchored on the TER axis. Cost is static; return is market noise."
    ),
    "06_cumulative_nav_growth": (
        f"Plot 6 · Bivariate. Over 24 months the median Direct-plan fund compounds to {cum_dir:.2f}% "
        f"cumulative growth vs {cum_reg:.2f}% for the median Regular-plan fund. The {cum_gap:.2f} pp "
        f"gap is the visible compounding of the {median_reg_v - median_dir_v:.2f} pp/year fee."
    ),
    "07_all_funds_monthly_return": (
        f"Plot 7 · Univariate in time. The monthly returns of all {line_data['name_norm'].nunique()} "
        f"Direct-plan funds move together as a cloud, {line_data['ret_pct'].min():.1f}% to "
        f"{line_data['ret_pct'].max():.1f}%. Fund-specific drift is not visible month to month — which "
        f"is why the return-vs-TER slope in Plot 4 is small."
    ),
    "08_return_and_ter_highlight": (
        f"Plot 8 · Bivariate on twin axes. One fund's return (solid) and Direct TER (dashed) are shown "
        f"against the faded cloud of all {fund_x.shape[0]} funds. The highlighted TER line is nearly flat; "
        f"the return line swings several percentage points — cost is stable, return is volatile."
    ),
    "09_sip_accumulated": (
        f"Plot 9 · Bivariate. A ₹{SIP_AMOUNT:,}/month SIP accumulates ₹{sip_inv:,.0f} of contributions "
        f"over 24 months. The Direct-plan median is ₹{sip_dir:,.0f} vs ₹{sip_reg:,.0f} for Regular — "
        f"₹{sip_gap:,.0f} ahead. Short window means small absolute gap; over 10 years the same gap compounds "
        f"to roughly ₹10,000 per ₹1,000/month."
    ),
    "00_before_after": (
        f"Plot 0 · Before vs after cleaning. The raw gap distribution contains {int((ba_before < 0).sum())} "
        f"negative-gap rows from unfiled-TER placeholders and a JioBlackRock artefact. After cleaning "
        f"the negative tail is gone (0 rows), and the median stays at {ba_after.median():.2f} pp — "
        f"cleaning removed defects without shifting the estimate."
    ),
}

# ---------- 4. Assemble the HTML ----------
def fig_block(png_path, caption):
    b64 = base64.b64encode(png_path.read_bytes()).decode()
    return (
        f'<figure>'
        f'<img src="data:image/png;base64,{b64}" alt="{png_path.stem}">'
        f'<figcaption>{caption}</figcaption>'
        f'</figure>'
    )

def section(num, heading, tag, png_name, interactive_html=None):
    png_path = PLOTS / f"{png_name}.png"
    if not png_path.exists():
        return f'<section data-num="{num}"><h2>{heading} <span class="tag">{tag}</span></h2><p>Missing: {png_name}.png</p></section>'
    extra = ""
    if interactive_html:
        extra = f'<p class="hint">Interactive: <a href="plots/{interactive_html}">{interactive_html}</a></p>'
    return (
        f'<section data-num="{num}">'
        f'<h2>{heading} <span class="tag">{tag}</span></h2>'
        f'{fig_block(png_path, captions[png_name])}'
        f'{extra}'
        f'</section>'
    )

sections_html = "".join([
    section(1, "TER distributions", "univariate · colour separates plans", "01_ter_distributions"),
    section(2, "Gap per fund", "univariate · sorted, one highlight", "02_gap_by_fund"),
    section(3, "Gap over time", "bivariate · stability check", "03_gap_over_time"),
    section(4, "Return vs TER", "bivariate · OLS fit", "04_return_vs_ter"),
    section(5, "Bubble over time", "animated · 24 frames", "05_bubble_over_time",
            interactive_html="05_bubble_over_time.html"),
    section(6, "Cumulative NAV growth", "bivariate · compounding", "06_cumulative_nav_growth"),
    section(7, "Every fund's monthly return", "univariate in time", "07_all_funds_monthly_return"),
    section(8, "One fund highlighted", "bivariate · twin axes", "08_return_and_ter_highlight"),
    section(9, f"₹{SIP_AMOUNT:,}/month SIP", "bivariate · rupee value", "09_sip_accumulated"),
    section(0, "Before vs after cleaning", "data-quality proof", "00_before_after"),
])

html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Large-cap TER and the direct–regular gap, 2024-10 to 2026-09</title>
<style>
  :root {{
    --regular: #e67e22;
    --direct:  #1f3a93;
    --highlight: #c0392b;
    --quant:   #00b894;
    --neutral: #7f8c8d;
    --text:    #222;
    --muted:   #555;
    --bg:      #fbfbfc;
    --line:    #d9dee4;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    font-family: -apple-system, "Segoe UI", Helvetica, Arial, sans-serif;
    color: var(--text); background: var(--bg);
    max-width: 1080px; margin: 0 auto; padding: 40px 24px 80px;
    line-height: 1.55;
  }}
  header {{ border-bottom: 3px solid var(--direct); padding-bottom: 20px; margin-bottom: 32px; }}
  header h1 {{ margin: 0 0 8px; font-size: 26px; color: var(--direct); letter-spacing: -0.3px; }}
  header p  {{ margin: 0; color: var(--muted); font-size: 15px; }}

  .kpis {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin: 28px 0 40px; }}
  .kpi  {{ background: #fff; border: 1px solid var(--line); border-radius: 8px; padding: 16px 18px;
           box-shadow: 0 1px 3px rgba(0,0,0,0.04); }}
  .kpi .label {{ font-size: 12px; letter-spacing: 0.4px; text-transform: uppercase;
                 color: var(--muted); margin-bottom: 6px; }}
  .kpi .value {{ font-size: 22px; font-weight: 600; color: var(--direct); }}
  .kpi .hint  {{ font-size: 12px; color: var(--muted); margin-top: 4px; }}

  section {{ margin: 44px 0; }}
  section h2 {{
    font-size: 18px; color: var(--direct);
    border-bottom: 1px solid var(--line);
    padding-bottom: 8px; margin-bottom: 20px;
    display: flex; align-items: baseline; gap: 12px;
  }}
  section h2 .tag {{ font-size: 12px; color: var(--muted); font-weight: 400; letter-spacing: 0.4px; }}
  section h2::before {{
    content: attr(data-num);
    display: inline-block; min-width: 30px;
    font-size: 13px; font-weight: 700;
    color: #fff; background: var(--direct);
    text-align: center; border-radius: 4px; padding: 2px 8px;
  }}

  figure {{ margin: 0; }}
  figure img {{
    width: 100%; height: auto;
    border: 1px solid var(--line); border-radius: 8px; background: #fff;
  }}
  figcaption {{
    margin-top: 12px; font-size: 14px; color: var(--muted); line-height: 1.6;
  }}

  .lead {{ font-size: 16px; color: var(--text); margin-bottom: 28px; }}
  .lead strong {{ color: var(--direct); }}
  .hint {{ font-size: 13px; color: var(--muted); margin-top: 8px; }}

  footer {{ margin-top: 60px; padding-top: 20px; border-top: 1px solid var(--line);
            font-size: 13px; color: var(--muted); }}
  footer a {{ color: var(--direct); text-decoration: none; }}
  footer a:hover {{ text-decoration: underline; }}
</style>
</head>
<body>

<header>
  <h1>Large-cap equity funds: TER and the direct–regular gap</h1>
  <p>Analysis window 2024-10 → 2026-09 · {fund_x.shape[0]} funds · 24 months · both plans · AMFI NAV portal and AMFI TER Excel</p>
</header>

<p class="lead">
  Over the 24-month window the median large-cap Direct plan charged <strong>{median_reg_v - (median_reg_v - median_dir_v):.2f}%</strong> per year
  while its Regular counterpart charged <strong>{median_reg_v:.2f}%</strong> — a gap of
  <strong>{median_reg_v - median_dir_v:.2f} percentage points</strong>. Across the {fund_x.shape[0]} funds
  the gap ranges from <strong>{fund_x['gap_med'].min():.2f}</strong> to <strong>{fund_x['gap_med'].max():.2f}</strong> pp,
  and it does not trend over the window. The relationship between TER and return is statistically detectable
  but small; the gap itself is the robust finding, and it compounds.
</p>

<div class="kpis">
  <div class="kpi"><div class="label">Funds</div><div class="value">{fund_x.shape[0]}</div>
                   <div class="hint">Large-cap, both plans</div></div>
  <div class="kpi"><div class="label">Median Reg TER</div><div class="value">{median_reg_v:.2f}%</div>
                   <div class="hint">per year</div></div>
  <div class="kpi"><div class="label">Median Dir TER</div><div class="value">{median_dir_v:.2f}%</div>
                   <div class="hint">per year</div></div>
  <div class="kpi"><div class="label">Median gap</div><div class="value">{median_gap_v:.2f} pp</div>
                   <div class="hint">IQR {iqr_lo:.2f}–{iqr_hi:.2f} pp</div></div>
</div>

{sections_html}

<footer>
  <p><strong>Sources.</strong>
    AMFI NAV history —
      <a href="https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx">portal.amfiindia.com</a>
      (collected 24 Oct 2026).
    AMFI TER Excel via <code>amfipy</code> <code>client.ter.download_excel</code>
      (collected 24 Oct 2026).
    Fund Performance snapshot from
      <a href="https://www.amfiindia.com/otherdata/fund-performance">amfiindia.com/otherdata/fund-performance</a>
      (downloaded 24 Oct 2026).
  </p>
  <p><strong>Libraries.</strong> pandas, numpy, scipy, plotly, plotly.express, dash, dash-bootstrap-components, amfipy, pyarrow.</p>
  <p><strong>AI assistance.</strong> ChatGPT was used for pipeline design, code scaffolding, and drafting this page; every number was verified by running the notebooks.</p>
</footer>

</body>
</html>
"""

(CLEAN / "dashboard.html").write_text(html, encoding="utf-8")
print("wrote clean/dashboard.html")
print("size:", (CLEAN / "dashboard.html").stat().st_size, "bytes")
print("plots referenced:", len(list(PLOTS.glob("*.png"))))
print("Open clean/dashboard.html in a browser — no server needed.")

cumulative return at end: Regular -3.07% | Direct -1.06% | gap 2.00 pp
SIP at end: invested ₹24,000 | Regular ₹23,139 | Direct ₹23,388 | gap ₹248
wrote clean/dashboard.html
size: 2143910 bytes
plots referenced: 10
Open clean/dashboard.html in a browser — no server needed.


In [53]:
# ============================================================
# FINAL DASHBOARD — one cell, all interactive figures, one HTML
# ============================================================
from pathlib import Path
import base64, re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

ROOT = Path("..").resolve()
RAW   = ROOT / "raw"
CLEAN = ROOT / "clean"
LOGS  = ROOT / "logs"
PLOTS = CLEAN / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

COLORS = {
    "regular":   "#e67e22",
    "direct":    "#1f3a93",
    "highlight": "#c0392b",
    "neutral":   "#7f8c8d",
}
QUANT_COLOR = "#00b894"
TEMPLATE    = "plotly_white"
SIP_AMOUNT  = 1000

# ---------- data ----------
panel = pd.read_parquet(CLEAN / "largecap_panel.parquet")
panel["month_dt"] = pd.to_datetime(panel["month"] + "-01")

fund_x = (panel.groupby("name_norm")
               .agg(reg_ter_med=("reg_ter", "median"),
                    dir_ter_med=("dir_ter", "median"),
                    gap_med=("gap", "median"),
                    ret_mean=("monthly_return", "mean"))
               .reset_index())
median_reg = float(panel["reg_ter"].median())
median_dir = float(panel["dir_ter"].median())
median_gap = float(panel["gap"].median())

# ---------- shared layout ----------
def base_layout(title, xtitle, ytitle, height=380):
    return dict(template=TEMPLATE, title=title,
                xaxis_title=xtitle, yaxis_title=ytitle,
                margin=dict(l=60, r=30, t=60, b=50),
                height=height, hovermode="closest")

# =========================================================
# Plot 1 · TER distributions
# =========================================================
fig1 = go.Figure()
fig1.add_trace(go.Histogram(x=panel["reg_ter"], name="Regular",
                            marker_color=COLORS["regular"], opacity=0.75, nbinsx=40))
fig1.add_trace(go.Histogram(x=panel["dir_ter"], name="Direct",
                            marker_color=COLORS["direct"], opacity=0.85, nbinsx=40))
fig1.add_vline(x=median_reg, line_dash="dash", line_color=COLORS["regular"], line_width=2)
fig1.add_vline(x=median_dir, line_dash="dash", line_color=COLORS["direct"], line_width=2)
fig1.update_layout(**base_layout(
    f"TER distributions — Direct is {median_reg - median_dir:.2f} pp cheaper",
    "Total Expense Ratio (% per year)", "Count of scheme-months",
))
fig1.update_layout(barmode="overlay", showlegend=True,
                   legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"))

# =========================================================
# Plot 2 · Gap per fund
# =========================================================
gap_by_fund = fund_x.sort_values("gap_med").reset_index(drop=True)
gap_by_fund["short"] = (gap_by_fund["name_norm"]
                        .str.replace(" fund", "", regex=False)
                        .str.replace("large cap", "LC", regex=False)
                        .str.title())
colors_gap = [COLORS["direct"]] * len(gap_by_fund)
colors_gap[-1] = COLORS["highlight"]
fig2 = go.Figure(go.Bar(
    x=gap_by_fund["gap_med"], y=gap_by_fund["short"], orientation="h",
    marker_color=colors_gap,
    hovertemplate="%{y}<br>median gap: %{x:.2f} pp<extra></extra>",
))
fig2.update_layout(**base_layout(
    f"Median gap per fund — {gap_by_fund['gap_med'].min():.2f} to "
    f"{gap_by_fund['gap_med'].max():.2f} pp",
    "Median gap (pp per year)", "", height=520,
))
fig2.update_layout(margin=dict(l=180, r=30, t=60, b=50), showlegend=False)

# =========================================================
# Plot 3 · Gap over time
# =========================================================
gap_month = (panel.groupby("month_dt")
                  .agg(gap_med=("gap", "median"),
                       p25=("gap", lambda s: s.quantile(0.25)),
                       p75=("gap", lambda s: s.quantile(0.75)))
                  .reset_index())
peak = gap_month.loc[gap_month["gap_med"].idxmax()]
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=pd.concat([gap_month["month_dt"], gap_month["month_dt"][::-1]]),
    y=pd.concat([gap_month["p75"], gap_month["p25"][::-1]]),
    fill="toself", fillcolor="rgba(31,58,147,0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    name="IQR", hoverinfo="skip",
))
fig3.add_trace(go.Scatter(
    x=gap_month["month_dt"], y=gap_month["gap_med"],
    mode="lines+markers",
    line=dict(color=COLORS["direct"], width=2.5), marker=dict(size=7),
    name="Median gap",
    hovertemplate="%{x|%Y-%m}<br>%{y:.2f} pp<extra></extra>",
))
fig3.add_trace(go.Scatter(
    x=[peak["month_dt"]], y=[peak["gap_med"]],
    mode="markers", marker=dict(color=COLORS["highlight"], size=12),
    name=f"Peak {peak['gap_med']:.2f}",
    hovertemplate="%{x|%Y-%m}<br>%{y:.2f} pp<extra></extra>",
))
fig3.update_layout(**base_layout(
    f"Gap over time — stable {gap_month['gap_med'].min():.2f}–"
    f"{gap_month['gap_med'].max():.2f} pp",
    "Month", "Median gap (pp per year)",
))
fig3.update_layout(legend=dict(x=0.01, y=0.01, xanchor="left", yanchor="bottom"))

# =========================================================
# Plot 4 · Return vs TER
# =========================================================
xv = fund_x["reg_ter_med"].values
yv = fund_x["ret_mean"].values
slope, intercept, r, p, se = stats.linregress(xv, yv)
fund_agg_plot = pd.DataFrame({
    "name_norm":    fund_x["name_norm"],
    "reg_ter_med":  fund_x["reg_ter_med"],
    "ret_mean_pct": fund_x["ret_mean"] * 100,
})
fig4 = px.scatter(
    fund_agg_plot, x="reg_ter_med", y="ret_mean_pct", hover_name="name_norm",
    labels={"reg_ter_med": "Median Regular TER (% per year)",
            "ret_mean_pct": "Mean monthly return (%)"},
    template=TEMPLATE,
    title=f"Return vs TER — slope {slope*100:.2f} pp, R² {r**2:.2f}, p {p:.3f}",
)
fig4.update_traces(marker=dict(size=11, color=COLORS["direct"],
                               line=dict(width=1, color="white")))
xx = np.linspace(xv.min(), xv.max(), 100)
fig4.add_trace(go.Scatter(
    x=xx, y=(intercept + slope * xx) * 100,
    mode="lines", line=dict(color=COLORS["highlight"], width=2.5),
    name=f"OLS (slope {slope*100:.2f})", hoverinfo="skip",
))
fig4.update_layout(**base_layout(
    f"Return vs TER — slope {slope*100:.2f}, R² {r**2:.2f}",
    "Median Regular TER (% per year)", "Mean monthly return (%)",
))

# =========================================================
# Plot 5 · Bubble (Direct only, fixed-size circles)
# =========================================================
aum_file = RAW / "amfi_fund_performance_largecap_2026-10-24.xlsx"
aum = pd.read_excel(aum_file, sheet_name="Fund_Performance", header=4)
aum.columns = [str(c).strip() for c in aum.columns]
aum["Daily AUM (Cr.)"] = (aum["Daily AUM (Cr.)"].astype(str)
                         .str.replace(",", "", regex=False)
                         .str.replace("^", "", regex=False)
                         .str.strip()
                         .replace({"": None, "nan": None})
                         .pipe(pd.to_numeric, errors="coerce"))
aum = aum[aum["Scheme Name"].notna()].copy()
aum = aum[aum["Scheme Name"].str.contains("Large Cap|Largecap", case=False, na=False)].copy()

def short_norm(s):
    if not isinstance(s, str): return ""
    s = s.lower()
    s = s.replace("(formerly", " (formerly").split(" (formerly")[0]
    s = s.replace("(erstwhile", " (erstwhile").split(" (erstwhile")[0]
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    s = re.sub(r"\blargecap\b", "large cap", s)
    s = re.sub(r"\b(plan|option|direct|regular|growth|idcw|dividend|payout|"
               r"reinvestment|bonus|withdrawal)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

aum["name_norm"] = aum["Scheme Name"].apply(short_norm)
aum_per_fund = (aum.groupby("name_norm")["Daily AUM (Cr.)"].max()
                  .reset_index().rename(columns={"Daily AUM (Cr.)": "aum_cr"}))
panel2 = panel.merge(aum_per_fund, on="name_norm", how="left")
bubble = (panel2.dropna(subset=["monthly_return", "dir_ter"])
                .query("plan == 'Direct'").copy())
bubble["ret_pct"]    = bubble["monthly_return"] * 100
bubble["ter_direct"] = bubble["dir_ter"]

x_rng = [float(bubble["ter_direct"].min() * 0.9), float(bubble["ter_direct"].max() * 1.1)]
y_rng = [float(bubble["ret_pct"].quantile(0.02)), float(bubble["ret_pct"].quantile(0.98))]

fig5 = px.scatter(
    bubble, x="ter_direct", y="ret_pct",
    hover_name="NAV_Name", hover_data={"aum_cr": ":.0f"},
    animation_frame="month", animation_group="name_norm",
    range_x=x_rng, range_y=y_rng,
    color_discrete_sequence=[COLORS["direct"]],
    opacity=0.6,
    labels={"ter_direct": "Direct TER (% per year)",
            "ret_pct":    "Monthly return (%)",
            "aum_cr":     "AUM (₹ Cr.)"},
    template=TEMPLATE,
    title="Direct TER vs monthly return — animated",
)
fig5.update_traces(marker=dict(size=9))
fig5.update_layout(height=480, margin=dict(l=60, r=30, t=60, b=50),
                   updatemenus=[dict(
                       type="buttons", showactive=False,
                       x=0.0, y=-0.12, xanchor="left", yanchor="top",
                       buttons=[
                           dict(label="▶ Play", method="animate",
                                args=[None, {"frame": {"duration": 700, "redraw": True},
                                             "transition": {"duration": 300},
                                             "fromcurrent": True}]),
                           dict(label="⏸ Pause", method="animate",
                                args=[[None], {"frame": {"duration": 0, "redraw": False},
                                               "transition": {"duration": 0},
                                               "mode": "immediate"}]),
                       ],
                   )])

# =========================================================
# Plot 6 · Cumulative NAV growth
# =========================================================
pg = panel.sort_values(["name_norm", "plan", "month"]).copy()
pg["mi"] = pg.groupby(["name_norm", "plan"]).cumcount()
pg["nb"] = pg.groupby(["name_norm", "plan"])["nav_last"].transform("first")
pg["cum"] = (pg["nav_last"] / pg["nb"] - 1) * 100

agg_cum = (pg.groupby(["mi", "month", "plan"])["cum"]
             .agg(median="median",
                  p25=lambda s: s.quantile(0.25),
                  p75=lambda s: s.quantile(0.75))
             .reset_index().sort_values("mi"))
agg_cum["month_dt"] = pd.to_datetime(agg_cum["month"] + "-01")

fig6 = go.Figure()
for pname, colour, fill in [
    ("Regular", COLORS["regular"], "rgba(230,126,34,0.15)"),
    ("Direct",  COLORS["direct"],  "rgba(31,58,147,0.15)"),
]:
    sub = agg_cum[agg_cum["plan"] == pname]
    fig6.add_trace(go.Scatter(
        x=pd.concat([sub["month_dt"], sub["month_dt"][::-1]]),
        y=pd.concat([sub["p75"], sub["p25"][::-1]]),
        fill="toself", fillcolor=fill,
        line=dict(color="rgba(255,255,255,0)"),
        name=f"{pname} IQR", hoverinfo="skip", showlegend=False,
    ))
    fig6.add_trace(go.Scatter(
        x=sub["month_dt"], y=sub["median"],
        mode="lines+markers",
        line=dict(color=colour, width=2.5), marker=dict(size=6),
        name=f"{pname} median",
        hovertemplate=f"{pname}<br>%{{x|%Y-%m}}<br>%{{y:.2f}}%<extra></extra>",
    ))
final_reg_row = agg_cum[agg_cum["plan"] == "Regular"].iloc[-1]
final_dir_row = agg_cum[agg_cum["plan"] == "Direct"].iloc[-1]
cum_gap = float(final_dir_row["median"] - final_reg_row["median"])
fig6.update_layout(**base_layout(
    f"Cumulative NAV growth — Direct ahead by {cum_gap:.2f} pp",
    "Month", "Cumulative NAV growth (%)",
))
fig6.update_layout(legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"))

# =========================================================
# Plot 7 · All funds monthly return (Direct)
# =========================================================
ret_data = (panel.query("plan == 'Direct'")
                 .dropna(subset=["monthly_return", "month_dt"])
                 .sort_values(["name_norm", "month_dt"]).copy())
ret_data["ret_pct"] = ret_data["monthly_return"] * 100
fund_list = sorted(ret_data["name_norm"].unique())
palette = px.colors.qualitative.Light24
fund_colours = {name: palette[i % len(palette)] for i, name in enumerate(fund_list)}

fig7 = go.Figure()
for name in fund_list:
    sub = ret_data[ret_data["name_norm"] == name]
    fig7.add_trace(go.Scatter(
        x=sub["month_dt"], y=sub["ret_pct"], mode="lines",
        line=dict(color=fund_colours[name], width=1.5), opacity=0.3,
        name=name, showlegend=False,
        hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}<br>%{{y:.2f}}%<extra></extra>",
    ))
fig7.update_layout(**base_layout(
    f"Monthly return, all {len(fund_list)} Direct-plan funds",
    "Month", "Monthly return (%)", height=420,
))

# =========================================================
# Plot 8 · One fund highlighted (return + TER, twin axes)
# =========================================================
HIGHLIGHT_FUND = "quant large cap fund"
line_data = (panel.query("plan == 'Direct'")
                  .dropna(subset=["monthly_return", "month_dt", "dir_ter"])
                  .sort_values(["name_norm", "month_dt"]).copy())
line_data["ret_pct"] = line_data["monthly_return"] * 100

fig8 = go.Figure()
for name in fund_list:
    sub = line_data[line_data["name_norm"] == name]
    is_sel = (name == HIGHLIGHT_FUND)
    fig8.add_trace(go.Scatter(
        x=sub["month_dt"], y=sub["ret_pct"],
        mode="lines+markers" if is_sel else "lines",
        line=dict(color=COLORS["highlight"] if is_sel else fund_colours[name],
                  width=3 if is_sel else 1),
        marker=dict(size=6) if is_sel else None,
        opacity=1.0 if is_sel else 0.10,
        name=name, showlegend=False, yaxis="y",
        hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}<br>return: %{{y:.2f}}%<extra></extra>",
    ))
    fig8.add_trace(go.Scatter(
        x=sub["month_dt"], y=sub["dir_ter"],
        mode="lines+markers" if is_sel else "lines",
        line=dict(color=COLORS["highlight"] if is_sel else fund_colours[name],
                  width=2 if is_sel else 1,
                  dash="dash" if is_sel else "dot"),
        marker=dict(size=5, symbol="square") if is_sel else None,
        opacity=1.0 if is_sel else 0.10,
        name=name, showlegend=False, yaxis="y2",
        hovertemplate=f"<b>{name}</b><br>%{{x|%Y-%m}}<br>TER: %{{y:.2f}}%<extra></extra>",
    ))
fig8.update_layout(
    template=TEMPLATE,
    title=f"Return (solid) and Direct TER (dashed) — {HIGHLIGHT_FUND} highlighted",
    xaxis_title="Month",
    yaxis=dict(title="Monthly NAV return (%)"),
    yaxis2=dict(title="Direct TER (%)", overlaying="y", side="right", showgrid=False),
    margin=dict(l=70, r=80, t=60, b=50),
    height=460, hovermode="closest", showlegend=False,
)

# =========================================================
# Plot 9 · SIP accumulated value
# =========================================================
sip_base = (panel.dropna(subset=["nav_last", "month_dt"])
                 .sort_values(["name_norm", "plan", "month_dt"]))
sip_pieces = []
for (name, plan), grp in sip_base.groupby(["name_norm", "plan"], sort=False):
    g = grp.sort_values("month_dt").reset_index(drop=True).copy()
    g["units"] = SIP_AMOUNT / g["nav_last"]
    g["cum_units"] = g["units"].cumsum()
    g["cum_invested"] = SIP_AMOUNT * (g.index + 1)
    g["cum_value"] = g["cum_units"] * g["nav_last"]
    g["name_norm"] = name; g["plan"] = plan
    sip_pieces.append(g[["name_norm", "plan", "month_dt", "cum_invested", "cum_value"]])
sip_df = pd.concat(sip_pieces, ignore_index=True)

sip_agg = (sip_df.groupby(["month_dt", "plan"])
                .agg(inv=("cum_invested", "median"),
                     val=("cum_value", "median"))
                .reset_index().sort_values("month_dt"))

fig9 = go.Figure()
inv = sip_agg[sip_agg["plan"] == "Regular"]
fig9.add_trace(go.Scatter(
    x=inv["month_dt"], y=inv["inv"], mode="lines",
    line=dict(color=COLORS["neutral"], width=2.5, dash="dot"),
    name="Invested (₹1,000/month)",
    hovertemplate="%{x|%Y-%m}<br>invested: ₹%{y:,.0f}<extra></extra>",
))
for pname, colour in [("Regular", COLORS["regular"]), ("Direct", COLORS["direct"])]:
    sub = sip_agg[sip_agg["plan"] == pname]
    fig9.add_trace(go.Scatter(
        x=sub["month_dt"], y=sub["val"], mode="lines+markers",
        line=dict(color=colour, width=2.5), marker=dict(size=6),
        name=f"{pname} value",
        hovertemplate=f"{pname}<br>%{{x|%Y-%m}}<br>₹%{{y:,.0f}}<extra></extra>",
    ))
sip_reg = float(sip_agg[(sip_agg["plan"] == "Regular")].iloc[-1]["val"])
sip_dir = float(sip_agg[(sip_agg["plan"] == "Direct")].iloc[-1]["val"])
sip_inv = float(sip_agg[(sip_agg["plan"] == "Regular")].iloc[-1]["inv"])
sip_gap = sip_dir - sip_reg

fig9.update_layout(**base_layout(
    f"₹{SIP_AMOUNT:,}/month SIP — Direct ends ₹{sip_gap:,.0f} ahead",
    "Month", "Value (₹)", height=460,
))
fig9.update_layout(legend=dict(x=0.01, y=0.99, xanchor="left", yanchor="top"))

# =========================================================
# Plot 0 · Before / after cleaning
# =========================================================
ter_raw = pd.read_parquet(ROOT / "interim" / "ter_usable_2024-10_2026-09.parquet")
ter_raw = ter_raw[ter_raw["NSDL Scheme Code"].isin(set(panel["NSDL_code"].dropna()))].copy()
ter_raw["reg_ter"] = pd.to_numeric(ter_raw["Regular Plan - Total TER (%)"], errors="coerce")
ter_raw["dir_ter"] = pd.to_numeric(ter_raw["Direct Plan - Total TER (%)"], errors="coerce")
before = (ter_raw["reg_ter"] - ter_raw["dir_ter"]).dropna()
after  = panel["gap"].dropna()

fig0 = go.Figure()
fig0.add_trace(go.Histogram(x=before, name=f"Before ({len(before)})",
                            marker_color=COLORS["regular"], opacity=0.6, nbinsx=60))
fig0.add_trace(go.Histogram(x=after,  name=f"After ({len(after)})",
                            marker_color=COLORS["direct"], opacity=0.85, nbinsx=60))
fig0.add_vline(x=0, line_dash="dot", line_color=COLORS["neutral"])
fig0.update_layout(**base_layout(
    f"Before vs after cleaning — {int((before < 0).sum())} negative rows removed",
    "Direct–regular gap (pp per year)", "Count of scheme-months",
))
fig0.update_layout(barmode="overlay", showlegend=True,
                   legend=dict(x=0.99, y=0.99, xanchor="right", yanchor="top"))

# =========================================================
# Write the HTML — every figure is a live Plotly chart
# =========================================================
FIGS = [
    ("0", "Before vs after cleaning", "data-quality proof", fig0,
     f"The raw gap distribution contains {int((before < 0).sum())} negative-gap rows from "
     f"unfiled-TER placeholders and a JioBlackRock artefact. After cleaning the negative tail is "
     f"gone and the median stays at {after.median():.2f} pp — the cleaning removed defects without "
     f"moving the estimate."),
    ("1", "TER distributions", "univariate · colour separates plans", fig1,
     f"Median Regular TER is {median_reg:.2f}% per year; median Direct is {median_dir:.2f}%. "
     f"The horizontal offset of {median_reg - median_dir:.2f} pp between the two distributions "
     f"is the direct–regular cost gap. Orange vs navy survive greyscale printing."),
    ("2", "Gap per fund", "univariate · sorted, one highlight", fig2,
     f"Median per-fund gap ranges from {fund_x['gap_med'].min():.2f} pp to "
     f"{fund_x['gap_med'].max():.2f} pp across {fund_x.shape[0]} funds, a "
     f"{fund_x['gap_med'].max() / fund_x['gap_med'].min():.1f}× spread. The gap is not one number."),
    ("3", "Gap over time", "bivariate · stability", fig3,
     f"The monthly median gap stays between {gap_month['gap_med'].min():.2f} and "
     f"{gap_month['gap_med'].max():.2f} pp across the 24-month window. No trend, no widening — a "
     f"structural feature of the category."),
    ("4", "Return vs TER", "bivariate · OLS fit", fig4,
     f"Regression of fund mean monthly return on median Regular TER gives slope "
     f"{slope*100:.2f} pp per pp, R² = {r**2:.2f}, p = {p:.3f} on {len(xv)} funds. Detectable but "
     f"small — monthly return variance is dominated by market beta."),
    ("5", "Bubble over time", "animated · 24 frames", fig5,
     "Direct-plan bubbles drift on the return axis across 24 monthly frames but stay anchored on "
     "the TER axis. Cost is static; return is market noise."),
    ("6", "Cumulative NAV growth", "bivariate · compounding", fig6,
     f"Over 24 months the median Direct-plan fund compounds to {final_dir_row['median']:.2f}% "
     f"cumulative growth vs {final_reg_row['median']:.2f}% for the Regular-plan median. The "
     f"{cum_gap:.2f} pp gap is the visible compounding of the {median_reg - median_dir:.2f} "
     f"pp/year fee."),
    ("7", "All funds monthly return", "univariate in time", fig7,
     f"The monthly returns of all {len(fund_list)} Direct-plan funds move together as a cloud, "
     f"{ret_data['ret_pct'].min():.1f}% to {ret_data['ret_pct'].max():.1f}%. Fund-specific drift is "
     f"not visible month to month — which is why the return-vs-TER slope is small."),
    ("8", "One fund highlighted", "bivariate · twin axes", fig8,
     f"One fund's monthly return (solid) and Direct TER (dashed) overlaid on the faded cloud of all "
     f"{len(fund_list)} Direct-plan funds. The TER line is nearly flat; the return line swings "
     f"several percentage points."),
    ("9", f"₹{SIP_AMOUNT:,}/month SIP", "bivariate · rupee value", fig9,
     f"Over 24 months, ₹{SIP_AMOUNT:,}/month SIP accumulates ₹{sip_inv:,.0f} of contributions. The "
     f"Direct-plan median is ₹{sip_dir:,.0f} vs ₹{sip_reg:,.0f} for Regular — ₹{sip_gap:,.0f} "
     f"ahead. Short window, small absolute gap; over 10 years the same gap compounds to about "
     f"₹10,000 per ₹1,000/month."),
]

# Convert each figure to an inline HTML div with Plotly's CDN script
div_html = []
for num, title, tag, fig, caption in FIGS:
    fig_html = pio.to_html(
        fig, include_plotlyjs=("cdn" if not div_html else False),
        full_html=False, config={"displaylogo": False, "responsive": True},
    )
    div_html.append(f"""
<section>
  <h2 data-num="{num}">{title} <span class="tag">{tag}</span></h2>
  <div class="plot">{fig_html}</div>
  <p class="cap">{caption}</p>
</section>
""")

html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Large-cap TER and the direct–regular gap · 2024-10 → 2026-09</title>
<style>
  :root {{
    --regular:   #e67e22;
    --direct:    #1f3a93;
    --highlight: #c0392b;
    --neutral:   #7f8c8d;
    --text: #222; --muted:#555; --bg:#fbfbfc; --line:#d9dee4;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    font-family: -apple-system, "Segoe UI", Helvetica, Arial, sans-serif;
    color: var(--text); background: var(--bg);
    max-width: 1180px; margin: 0 auto; padding: 40px 24px 80px;
    line-height: 1.55;
  }}
  header {{ border-bottom: 3px solid var(--direct); padding-bottom: 20px; margin-bottom: 28px; }}
  header h1 {{ margin: 0 0 8px; font-size: 26px; color: var(--direct); letter-spacing: -0.3px; }}
  header p  {{ margin: 0; color: var(--muted); font-size: 15px; }}
  .lead {{ font-size: 16px; margin-bottom: 28px; }}
  .lead strong {{ color: var(--direct); }}
  .kpis {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin: 24px 0 40px; }}
  .kpi  {{ background:#fff; border:1px solid var(--line); border-radius:8px; padding:16px 18px;
           box-shadow: 0 1px 3px rgba(0,0,0,0.04); }}
  .kpi .label {{ font-size: 12px; letter-spacing: 0.4px; text-transform: uppercase;
                 color: var(--muted); margin-bottom: 6px; }}
  .kpi .value {{ font-size: 22px; font-weight: 600; color: var(--direct); }}
  .kpi .hint  {{ font-size: 12px; color: var(--muted); margin-top: 4px; }}
  .grid {{
    display: grid; grid-template-columns: 1fr 1fr; gap: 24px;
  }}
  @media (max-width: 900px) {{ .grid {{ grid-template-columns: 1fr; }} .kpis {{ grid-template-columns: 1fr 1fr; }} }}
  section {{ margin: 0; background: #fff; border: 1px solid var(--line); border-radius: 10px;
             padding: 18px 18px 8px; box-shadow: 0 1px 4px rgba(0,0,0,0.04); }}
  section h2 {{
    font-size: 16px; color: var(--direct); margin: 0 0 12px;
    display: flex; align-items: baseline; gap: 10px;
  }}
  section h2 .tag {{ font-size: 12px; color: var(--muted); font-weight: 400; letter-spacing: 0.3px; }}
  section h2::before {{
    content: attr(data-num); display: inline-block; min-width: 26px;
    font-size: 12px; font-weight: 700; color: #fff; background: var(--direct);
    text-align: center; border-radius: 4px; padding: 1px 8px;
  }}
  .plot {{ width: 100%; }}
  .cap  {{ font-size: 13px; color: var(--muted); margin: 8px 2px 14px; line-height: 1.55; }}
  footer {{ margin-top: 48px; padding-top: 20px; border-top: 1px solid var(--line);
            font-size: 13px; color: var(--muted); }}
  footer a {{ color: var(--direct); text-decoration: none; }}
  footer a:hover {{ text-decoration: underline; }}
</style>
</head>
<body>
<header>
  <h1>Large-cap equity funds: TER and the direct–regular gap</h1>
  <p>Analysis window 2024-10 → 2026-09 · {fund_x.shape[0]} funds · 24 months · both plans · AMFI NAV portal and AMFI TER Excel</p>
</header>

<p class="lead">
  Over the 24-month window the median large-cap Direct plan charged
  <strong>{median_dir:.2f}%</strong> per year while the median Regular plan charged
  <strong>{median_reg:.2f}%</strong> — a gap of
  <strong>{median_reg - median_dir:.2f} percentage points</strong>. Across the
  {fund_x.shape[0]} funds the gap ranges from
  <strong>{fund_x['gap_med'].min():.2f}</strong> to <strong>{fund_x['gap_med'].max():.2f}</strong> pp
  and does not trend over the window. The relationship between TER and return is statistically
  detectable but small; the gap itself is the robust finding, and it compounds.
</p>

<div class="kpis">
  <div class="kpi"><div class="label">Funds</div><div class="value">{fund_x.shape[0]}</div>
                   <div class="hint">Large-cap, both plans</div></div>
  <div class="kpi"><div class="label">Median Reg TER</div><div class="value">{median_reg:.2f}%</div>
                   <div class="hint">per year</div></div>
  <div class="kpi"><div class="label">Median Dir TER</div><div class="value">{median_dir:.2f}%</div>
                   <div class="hint">per year</div></div>
  <div class="kpi"><div class="label">Median gap</div><div class="value">{median_gap:.2f} pp</div>
                   <div class="hint">IQR {panel['gap'].quantile(.25):.2f}–{panel['gap'].quantile(.75):.2f} pp</div></div>
</div>

<div class="grid">
{''.join(div_html)}
</div>

<footer>
  <p><strong>Sources.</strong>
    AMFI NAV history — <a href="https://portal.amfiindia.com/DownloadNAVHistoryReport_Po.aspx">portal.amfiindia.com</a>
    (collected 24 Oct 2026).
    AMFI TER Excel via <code>amfipy</code> <code>client.ter.download_excel</code> (collected 24 Oct 2026).
    Fund Performance snapshot from <a href="https://www.amfiindia.com/otherdata/fund-performance">amfiindia.com/otherdata/fund-performance</a>
    (downloaded 24 Oct 2026).</p>
  <p><strong>Libraries.</strong> pandas, numpy, scipy, plotly, plotly.express, dash, dash-bootstrap-components, amfipy, pyarrow.</p>
  <p><strong>AI assistance.</strong> ChatGPT was used for pipeline design, code scaffolding, and drafting this page; every number was verified by running the notebooks.</p>
</footer>
</body>
</html>
"""

(CLEAN / "dashboard.html").write_text(html, encoding="utf-8")
size_mb = (CLEAN / "dashboard.html").stat().st_size / 1e6
print(f"wrote clean/dashboard.html  ({size_mb:.1f} MB)  —  {len(FIGS)} interactive plots on a 2-column grid")
print("Open clean/dashboard.html in a browser. No server, no internet after the first load.")

wrote clean/dashboard.html  (0.4 MB)  —  10 interactive plots on a 2-column grid
Open clean/dashboard.html in a browser. No server, no internet after the first load.
